<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/data_connectors/vietnamese_document_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vietnamese PDF ingestion for RAG

This example loads Vietnamese PDF documents with `SimpleDirectoryReader`, optionally
applies Vietnamese word segmentation **before** chunking, and builds a
`VectorStoreIndex` for retrieval.

It targets the current `llama-index-core` API. `Document.text` is a read-only
property backed by `text_resource`, so preprocessing writes content through
`Document.set_content(...)` instead of assigning to `doc.text`.

No retrieval-quality numbers are claimed here. The ~15% figure mentioned in the
requesting issue is a report from that team's own pipeline; measure on your own
evaluation set before deciding whether segmentation helps your documents.

## Install

`llama-index-readers-file` supplies `PDFReader`, which reads PDFs with `pypdf`.
`underthesea` supplies the Vietnamese word segmenter. The Hugging Face embedding
integration downloads a model on first use.

In [ ]:
%pip install llama-index-core llama-index-readers-file llama-index-embeddings-huggingface underthesea

## 1. Load the PDFs

`SimpleDirectoryReader` maps the `.pdf` extension to `PDFReader` from
`llama-index-readers-file`. PDFs that embed Vietnamese fonts can lose combining
tone marks during extraction, so always print a sample of the extracted text
before indexing; if diacritics are missing, fix extraction (or OCR) first rather
than compensating downstream.

`filename_as_id=True` keeps document IDs stable across re-ingestion, and
`required_exts` keeps unrelated files out of the index.

In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

reader = SimpleDirectoryReader(
    input_dir="./solar_docs_vi",
    required_exts=[".pdf"],
    filename_as_id=True,
)
documents = reader.load_data()

print(f"loaded {len(documents)} documents")
print(documents[0].text[:300])

## 2. Optional: Vietnamese word segmentation

Vietnamese separates syllables with spaces, not words. A chunker that counts or
splits on whitespace can therefore cut a compound word in half.
`underthesea.word_tokenize(..., format="text")` returns a string in which the
syllables of one word are joined with `_`, e.g. `Thành phố` becomes `Thành_phố`.

`Document.text` is a property in `llama-index-core/llama_index/core/schema.py`;
assigning to it does not update the stored `text_resource`. Use `set_content` so
the same `Document` keeps its `id_`, `metadata` and metadata-exclusion settings.

In [ ]:
from underthesea import word_tokenize

USE_SEGMENTATION = True  # set to False to compare against raw chunking

if USE_SEGMENTATION:
    for doc in documents:
        doc.set_content(word_tokenize(doc.text, format="text"))

## 3. Embedding model and chunking

`Settings` supplies the embedding model and text splitter to
`VectorStoreIndex.from_documents`. A Vietnamese-specific model is used below;
`intfloat/multilingual-e5-small` is a multilingual alternative, but check its
model card, because the e5 family expects `query:` / `passage:` prefixes.

Whichever model you choose, it must handle the segmented surface form. Keep the
segmentation step in both ingestion and query paths so that documents and
queries are embedded from text with the same shape.

In [ ]:
from llama_index.core import Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(
    model_name="keepitreal/vietnamese-sbert"
)
Settings.text_splitter = SentenceSplitter(chunk_size=512, chunk_overlap=64)

## 4. Index and retrieve

A retriever is used instead of a query engine so the example needs no LLM API
key. Apply the same segmentation to the question, otherwise the query is
embedded in a different surface form from the indexed chunks.

In [ ]:
index = VectorStoreIndex.from_documents(documents)

question = "Công suất lắp đặt của dự án là bao nhiêu?"
if USE_SEGMENTATION:
    question = word_tokenize(question, format="text")

for hit in index.as_retriever(similarity_top_k=3).retrieve(question):
    print(round(hit.score or 0.0, 4), hit.node.text[:200])

## Evaluating whether segmentation helps

Run the notebook twice (`USE_SEGMENTATION = True` and `False`) against the same
fixed set of questions, and compare retrieval hit rate or nDCG on your own
labelled data. The two runs must also use the identical query transformation,
embedding model and chunk size, otherwise the comparison is not controlled.